# Fill-in-the-blank: KNN gesture classification

This worksheet uses the same fixed 40 captures as the filtering worksheet. It keeps the real project geometry: 10 captures per class, 8 windows per capture, 19 raw rows per window, 24 features, normalized K equals 3, and a two-frame label stabilizer.

## Instructor flow

Learners fill each exercise in order. The project imports, constants, data, and target shapes are provided. Serial reading, user prompts, timestamp resampling, and profile-file saving are excluded so the focus stays on the KNN pipeline.

In [ ]:
from collections import Counter
from pathlib import Path
import sys
import numpy as np

def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "calibration_pipeline" / "eflesh_calibration" / "knn.py").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from inside the project repository.")

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "calibration_pipeline"))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

from eflesh_calibration.app import (
    CALIBRATION_CLEAN_SAMPLES, CALIBRATION_EDGE_SAMPLES,
    LabelStabilizer, STABILITY_FRAMES, TRAINING_WINDOW_STRIDE_SAMPLES,
)
from eflesh_calibration.knn import (
    CLASSES, FEATURE_COUNT, FILTER_SAMPLES, K, RAW_WINDOW_SAMPLES,
    WINDOW_SAMPLES, feature, make_model, normalized_distances, predict,
)
from teaching_samples import load_teaching_dataset

data = load_teaching_dataset()
captures = data["captures"]
capture_labels = data["labels"]
capture_numbers = data["capture_number"]
baseline = data["baseline"]

print("captures:", captures.shape)
print("classes:", CLASSES)
print("pipeline:", RAW_WINDOW_SAMPLES, "raw rows →", WINDOW_SAMPLES, "filtered rows →", FEATURE_COUNT, "features")

## Exercise 1 — extract eight windows from one capture

Finish the data-only version of the index logic in read_calibration_windows.

Targets: return 8 windows, each with shape (19, 12).

Hints:

- first usable smoothed index is the clean-center edge;
- final start is clean-center start plus clean length minus filtered window length;
- starts advance by the training stride;
- raw start is four frames before smoothed start.

In [ ]:
def extract_training_windows(capture):
    windows = []

    # TODO: replace the first two None values.
    first_smoothed_index = None
    final_start = None

    # TODO: replace the three None values in range.
    for smoothed_start in range(None, None, None):
        # TODO: replace both None values.
        raw_start = None
        window = None
        windows.append(window)

    return windows

example_windows = extract_training_windows(captures[0])
assert len(example_windows) == 8
assert all(window.shape == (RAW_WINDOW_SAMPLES, 12) for window in example_windows)
print("windows per capture:", len(example_windows))

## Exercise 2 — make the 320 labeled feature rows

For every one of the 40 captures:

1. extract eight windows;
2. turn each window into features with feature(window, baseline);
3. add eight matching labels; and
4. add eight matching capture numbers.

Target: all_features has shape (320, 24).

In [ ]:
all_features = []
all_labels = []
all_capture_numbers = []

for capture, label, number in zip(captures, capture_labels, capture_numbers):
    windows = extract_training_windows(capture)

    # TODO: replace the three None values with lists of length 8.
    feature_rows = None
    label_rows = None
    number_rows = None

    all_features.extend(feature_rows)
    all_labels.extend(label_rows)
    all_capture_numbers.extend(number_rows)

all_features = np.stack(all_features)
all_labels = np.asarray(all_labels)
all_capture_numbers = np.asarray(all_capture_numbers)

assert all_features.shape == (320, FEATURE_COUNT)
assert len(all_labels) == 320 and len(all_capture_numbers) == 320
print("feature matrix:", all_features.shape)

## Exercise 3 — train and predict with the production functions

Hold out complete captures 8 and 9. Train on capture numbers below 8.

Targets:

- train_features has shape (256, 24)
- test_features has shape (64, 24)
- model has K equal to 3
- one prediction is produced per test row

Hints: masks are Boolean arrays; make_model needs the labels as a normal list; predict works on one feature row.

In [ ]:
# TODO: replace all None values.
train_mask = None
test_mask = None
train_features = None
train_labels = None
test_features = None
test_labels = None
model = None
predicted_labels = None

assert train_features.shape == (256, FEATURE_COUNT)
assert test_features.shape == (64, FEATURE_COUNT)
assert model["k"] == K == 3
assert len(predicted_labels) == len(test_labels)

accuracy = np.mean(predicted_labels == test_labels)
print(f"held-out accuracy: {accuracy:.1%}")

## Exercise 4 — inspect one K equals 3 vote

Choose a held-out spread example. Calculate normalized distances and keep the first three stable-sorted indices.

Hint: use next, normalized_distances, and np.argsort with kind equal to stable.

Target: nearest has length 3.

In [ ]:
# TODO: replace the four None values.
query_index = None
query = None
distances = None
nearest = None

assert len(nearest) == K
nearest_labels = [model["labels"][int(index)] for index in nearest]

print("expected:", test_labels[query_index])
print("program prediction:", predict(model, query))
print("nearest labels:", nearest_labels)
print("vote:", dict(Counter(nearest_labels)))

## Exercise 5 — apply the real label stabilizer

A published live label needs two matching raw KNN labels. Fill in the one call inside the loop.

Expected behavior: the first wrist_up does not publish yet, and a one-frame rest glitch does not replace wrist_up.

In [ ]:
raw_predictions = ["wrist_up", "wrist_up", "rest", "wrist_up", "wrist_up"]

stabilizer = LabelStabilizer()
published = []
for raw_label in raw_predictions:
    # TODO: replace None with one stabilizer method call.
    stable_label = None
    published.append(stable_label)

assert STABILITY_FRAMES == 2
print("raw:      ", raw_predictions)
print("published:", published)

## Done

You rebuilt a simplified version of the actual KNN path:

40 captures → 320 labeled features → standardized K equals 3 model → stabilized live label.

If a learner is stuck, open the matching function in knn.py or app.py. The goal is to understand your implementation, not a separate toy classifier.